<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B07%5D%20-%20Ingenieria_de_Variables_I/%5B01%5D%20-%20Notebooks/E5_Mean_Encoding_Bien_vs_Mal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E5 · Mean encoding: bien vs mal — Ingenieria de Variables I (bonus)

## Introduccion

El **target / mean encoding** sustituye cada categoria por la media del target. Es compacto y
potente para **alta cardinalidad** (como `comercio`), pero hecho a la ligera **sobreajusta**.

Aqui comparamos tres formas de hacerlo y medimos la diferencia **en datos nuevos**:

- **A) Sin smoothing** (media directa sobre todo el train): el caso ingenuo → overfit.
- **B) Smoothing a mano**: suavizamos las categorias raras hacia la media global.
- **C) `TargetEncoder` + turnos**: validacion cruzada interna (out-of-fold) + smoothing automatico.

La idea clave: miraremos el **hueco entre AUC de train y de test**. Cuanto mayor el hueco,
mayor el sobreajuste.

## Objetivos del ejercicio

- Ver por que el mean encoding ingenuo se memoriza las categorias raras.
- Implementar **smoothing** y entender su formula.
- Usar `TargetEncoder` de scikit-learn (out-of-fold) y comparar las tres variantes.

## Descripcion del dataset (fraude con tarjeta)

Trabajamos con un dataset **sintetico y reproducible** de transacciones con tarjeta.
Lo generamos dentro del propio notebook para que sea autocontenido en Colab.
Cada fila es una transaccion con estas variables:

| Variable | Tipo | Descripcion |
|---|---|---|
| `id_cliente` | id | Identificador del cliente |
| `edad` | numerica | Edad del cliente (con algunos huecos) |
| `monto` | numerica | Importe de la transaccion en euros (distribucion sesgada) |
| `pais` | categorica | Pais de la operacion: ES, DE, UK, FR, IT |
| `tipo_tarjeta` | categorica | debito / credito / prepago |
| `comercio` | categorica (ALTA cardinalidad) | Comercio donde se opera (COM_0000 ... COM_0299, con frecuencias muy desiguales) |
| `canal` | categorica | online / presencial (con algunos huecos) |
| `codigo_postal` | pseudo-numerica | Parece numero, pero NO tiene magnitud |
| `fecha` | fecha/hora | Momento de la transaccion |
| `es_fraude` | objetivo (0/1) | 1 si la transaccion fue fraudulenta |

> La probabilidad real de fraude se ha construido en funcion del monto, la hora,
> el canal, el pais y el comercio. Por eso, **las variables que creemos tendran
> senal de verdad** y veremos su efecto en las metricas.

### 1. Importar librerias necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score
import sklearn
print("Version de scikit-learn:", sklearn.__version__, "(TargetEncoder requiere >= 1.3)")

### 2. Datos y particion

In [ ]:
import numpy as np
import pandas as pd

def generar_datos_fraude(n=8000, semilla=42):
    # Dataset sintetico y REPRODUCIBLE de transacciones con tarjeta.
    # La probabilidad de fraude depende de variables reales (monto, hora,
    # canal, pais y comercio): asi la ingenieria de variables tiene senal de verdad.
    rng = np.random.default_rng(semilla)

    # --- Perfil de clientes ---
    n_clientes = 600
    gasto_medio_cliente = rng.lognormal(mean=3.2, sigma=0.5, size=n_clientes)
    edad_por_cliente = rng.integers(18, 80, size=n_clientes)
    id_cliente = rng.integers(0, n_clientes, size=n)

    # --- Comercios (ALTA cardinalidad, con frecuencias MUY desiguales) ---
    n_comercios = 300
    comercios = np.array([f"COM_{i:04d}" for i in range(n_comercios)])
    riesgo_comercio = rng.beta(1.2, 8.0, size=n_comercios)   # casi todos bajos, unos pocos altos
    peso_comercio = 1.0 / np.arange(1, n_comercios + 1)      # ley de potencias: muchos comercios raros
    peso_comercio = peso_comercio / peso_comercio.sum()
    idx_comercio = rng.choice(n_comercios, size=n, p=peso_comercio)

    # --- Categoricas de BAJA cardinalidad ---
    pais = rng.choice(["ES", "DE", "UK", "FR", "IT"], size=n, p=[0.60, 0.12, 0.10, 0.10, 0.08])
    tipo_tarjeta = rng.choice(["debito", "credito", "prepago"], size=n, p=[0.55, 0.40, 0.05])
    canal = rng.choice(["online", "presencial"], size=n, p=[0.45, 0.55])

    # --- Codigo postal (PSEUDO-numerica: parece numero, pero es categorica) ---
    cp_base = rng.choice([28001, 8001, 41001, 46001, 50001], size=n)
    codigo_postal = cp_base + rng.integers(0, 40, size=n)

    # --- Monto (distribucion sesgada con outliers) ---
    monto = rng.lognormal(mean=np.log(gasto_medio_cliente[id_cliente]), sigma=0.8)
    gigantes = rng.random(n) < 0.005                         # unas pocas compras enormes
    monto[gigantes] *= rng.uniform(20, 80, size=int(gigantes.sum()))
    monto = np.round(monto, 2)

    # --- Fecha y hora ---
    inicio = np.datetime64("2024-01-01T00:00")
    minutos = rng.integers(0, 365 * 24 * 60, size=n)
    fecha = pd.to_datetime(inicio + minutos.astype("timedelta64[m]"))
    hora = fecha.hour.to_numpy()
    dia_semana = fecha.dayofweek.to_numpy()

    # --- Probabilidad de fraude: la SENAL vive en estas variables ---
    logit = (
        -4.2
        + 0.45 * (np.log1p(monto) - np.log1p(monto).mean())
        + 1.8 * (hora < 6)
        + 0.7 * (canal == "online")
        + 0.5 * (pais != "ES")
        + 4.0 * riesgo_comercio[idx_comercio]
        + 0.3 * (dia_semana >= 5)
    )
    prob = 1.0 / (1.0 + np.exp(-logit))
    es_fraude = rng.binomial(1, prob)

    df = pd.DataFrame({
        "id_cliente": id_cliente,
        "edad": edad_por_cliente[id_cliente].astype(float),
        "monto": monto,
        "pais": pais,
        "tipo_tarjeta": tipo_tarjeta,
        "comercio": comercios[idx_comercio],
        "canal": canal,
        "codigo_postal": codigo_postal,
        "fecha": fecha,
        "es_fraude": es_fraude,
    })

    # Valores faltantes realistas (para practicar imputacion)
    df.loc[rng.random(n) < 0.05, "edad"] = np.nan
    df.loc[rng.random(n) < 0.03, "canal"] = np.nan
    return df

In [ ]:
df = generar_datos_fraude(n=8000, semilla=42)
train, test = train_test_split(df, test_size=0.3, random_state=0, stratify=df["es_fraude"])

# Trabajamos con la variable de ALTA cardinalidad
col = "comercio"
y_train = train["es_fraude"].to_numpy()
y_test = test["es_fraude"].to_numpy()
media_global = train["es_fraude"].mean()
print("Nº de comercios:", train[col].nunique(), "| media global de fraude:", round(media_global, 4))

### 3. Metodo A — SIN smoothing (media directa)

Calculamos la media de fraude por comercio con **todo el train** y la aplicamos al propio train.
Las categorias con muy pocas filas se memorizan (su media es 0.0 o 1.0), inflando el AUC de train.

In [ ]:
medias = train.groupby(col)["es_fraude"].mean()
A_train = train[col].map(medias).to_numpy()
A_test = test[col].map(medias).fillna(media_global).to_numpy()

auc_A_train = roc_auc_score(y_train, A_train)
auc_A_test = roc_auc_score(y_test, A_test)
print(f"[A] Sin smoothing -> AUC train: {auc_A_train:.3f} | AUC test: {auc_A_test:.3f}")
print(f"    Hueco train-test: {auc_A_train - auc_A_test:.3f}  (cuanto mayor, mas overfit)")

### 4. Metodo B — Smoothing a mano

Mezclamos la media de la categoria con la media global, dando mas peso a la global cuando hay
pocas observaciones:

`encoding = (n · media_categoria + m · media_global) / (n + m)`

con `m` = "fuerza" del suavizado (cuantas observaciones equivalen a la media global).

In [ ]:
m = 30
stats = train.groupby(col)["es_fraude"].agg(["mean", "count"])
suav = (stats["count"] * stats["mean"] + m * media_global) / (stats["count"] + m)

B_train = train[col].map(suav).to_numpy()
B_test = test[col].map(suav).fillna(media_global).to_numpy()

auc_B_train = roc_auc_score(y_train, B_train)
auc_B_test = roc_auc_score(y_test, B_test)
print(f"[B] Smoothing a mano (m={m}) -> AUC train: {auc_B_train:.3f} | AUC test: {auc_B_test:.3f}")
print(f"    Hueco train-test: {auc_B_train - auc_B_test:.3f}")

### 5. Metodo C — `TargetEncoder` + turnos (out-of-fold)

`TargetEncoder` de scikit-learn hace el trabajo bien: en `fit_transform(train)` usa
**validacion cruzada interna** (out-of-fold) para que ninguna fila vea su propia respuesta,
y aplica **smoothing automatico**. Para `test` usa `transform` con todo el train.

In [ ]:
te = TargetEncoder(target_type="binary", smooth="auto", cv=5)
C_train = te.fit_transform(train[[col]], y_train)[:, 0]   # out-of-fold internamente
C_test = te.transform(test[[col]])[:, 0]

auc_C_train = roc_auc_score(y_train, C_train)
auc_C_test = roc_auc_score(y_test, C_test)
print(f"[C] TargetEncoder + turnos -> AUC train: {auc_C_train:.3f} | AUC test: {auc_C_test:.3f}")
print(f"    Hueco train-test: {auc_C_train - auc_C_test:.3f}")

### 6. Comparativa final

In [ ]:
resumen = pd.DataFrame({
    "metodo": ["A) Sin smoothing", "B) Smoothing a mano", "C) TargetEncoder + turnos"],
    "AUC_train": [auc_A_train, auc_B_train, auc_C_train],
    "AUC_test": [auc_A_test, auc_B_test, auc_C_test],
})
resumen["hueco_overfit"] = (resumen["AUC_train"] - resumen["AUC_test"]).round(3)
resumen = resumen.round(3)
print(resumen.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(resumen))
ax.bar(x - 0.2, resumen["AUC_train"], width=0.4, label="AUC train", color="#e67e22")
ax.bar(x + 0.2, resumen["AUC_test"], width=0.4, label="AUC test", color="#2980b9")
ax.set_xticks(x)
ax.set_xticklabels(resumen["metodo"], rotation=10)
ax.set_ylabel("AUC")
ax.set_title("Mean encoding: train vs test (el hueco mide el sobreajuste)")
ax.legend()
plt.tight_layout()
plt.show()

### La regla de oro del target encoding

> Calcula la media **por turnos (out-of-fold)** y **suaviza** las categorias raras (smoothing).
> Asi la media es honesta: el modelo **generaliza** a datos nuevos en vez de memorizar el entrenamiento.

### Reflexion

1. ¿Por que el metodo A tiene el AUC de train mas alto y, sin embargo, no es el mejor?
2. ¿Que le pasa al `hueco_overfit` al aplicar smoothing y turnos?
3. ¿Como afecta el parametro `m` del smoothing a las categorias con pocas observaciones?
4. ¿Por que `TargetEncoder` necesita el target `y` en `fit_transform` pero no en `transform`?